In [1]:
import glob
import gzip
import nbformat
from nbconvert.preprocessors import CellExecutionError
from nbconvert.preprocessors import ExecutePreprocessor
import os
import shutil
import tensorflow as tf
import time

2025-02-19 18:10:58.520180: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-19 18:11:18.189122: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib:/usr/local/lib/x86_64-linux-gnu:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2025-02-19 18:11:18.239799: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such 

In [2]:
#---[ Change this to be the name of the notebook in the current working directory that you wish to run. ]-----------
NOTEBOOK_TO_RUN = '002c_Process_BigSNPR_plinkfiles.ipynb'

#---[ Change the following to 'Python' if you have a Python notebook, or to 'R' if you have an R notebook.
KERNEL = 'R'


In [3]:
TIMESTAMP_FILE_SUFFIX = time.strftime('_%Y%m%d_%H%M%S.ipynb')
OUTPUT_NOTEBOOK = NOTEBOOK_TO_RUN.replace('.ipynb', TIMESTAMP_FILE_SUFFIX)

print(f'Executed notebook will be written to filename "{OUTPUT_NOTEBOOK}" on the local disk and the workspace bucket.')


Executed notebook will be written to filename "002c_Process_BigSNPR_plinkfiles_20250219_181131.ipynb" on the local disk and the workspace bucket.


In [4]:
DATESTAMP = time.strftime('%Y%m%d')
HAIL_LOG_DIR_FOR_PROVENANCE = os.path.join(os.getenv('WORKSPACE_BUCKET'), 'hail-logs', DATESTAMP)

print(f'Hail logs, if any, will be copied to {HAIL_LOG_DIR_FOR_PROVENANCE}')


Hail logs, if any, will be copied to gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/hail-logs/20250219


In [5]:
def get_kernel(kernel):
    return 'ir' if kernel.lower() == 'r' else 'python3'

KERNEL_NAME = get_kernel(KERNEL)


In [ ]:
# See also https://nbconvert.readthedocs.io/en/latest/execute_api.html
with open(NOTEBOOK_TO_RUN) as f_in:
    nb = nbformat.read(f_in, as_version=4)
    ep = ExecutePreprocessor(timeout=-1, kernel_name=KERNEL_NAME)
    try:
        out = ep.preprocess(nb, {'metadata': {'path': ''}})
    except CellExecutionError:
        out = None
        print(f'''Error executing the notebook "{NOTEBOOK_TO_RUN}".
        See notebook "{OUTPUT_NOTEBOOK}" for the traceback.''')
    finally:
        with open(OUTPUT_NOTEBOOK, mode='w', encoding='utf-8') as f_out:
            nbformat.write(nb, f_out)
        # Save the executed notebook to the workspace bucket.
        output_notebook_path = os.path.join(os.getenv('WORKSPACE_BUCKET'), 'notebooks', OUTPUT_NOTEBOOK)
        tf.io.gfile.copy(src=OUTPUT_NOTEBOOK, dst=output_notebook_path)
        print(f'Wrote executed notebook to {output_notebook_path}')

# Save the hail logs, if any, to the workspace bucket.
for hail_log in glob.glob('hail*.log'):
    with open(hail_log, 'rb') as f_in:
        compressed_hail_log = f'{hail_log}.gz'
        with gzip.open(compressed_hail_log, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    hail_log_path = os.path.join(HAIL_LOG_DIR_FOR_PROVENANCE, compressed_hail_log)
    if not tf.io.gfile.exists(hail_log_path):
        tf.io.gfile.copy(src=compressed_hail_log, dst=hail_log_path)
        print(f'Wrote hail log to {hail_log_path}')


Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/asthma_PRS_weights.txt...
| [1 files][128.1 MiB/128.1 MiB]                                                
Operation completed over 1 objects/128.1 MiB.                                    
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/atrial_fibrilation_PRS_weights.txt...
/ [1 files][ 20.2 KiB/ 20.2 KiB]                                                
Operation completed over 1 objects/20.2 KiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/breast_cancer_PRS_weights.txt...
/ [1 files][ 36.6 KiB/ 36.6 KiB]                                                
Operation completed over 1 objects/36.6 KiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/ckd_PRS_weights.txt...
- [1 files][ 52.4 MiB/ 52.4 MiB]                                                
Operation complete